# TF-IDF

- 문서에서 특정 단어가 얼마나 중요한지를 수치로 표현하는 방법

## TF
- 특정 단어가 문서에 얼마나 자주 등장하는지를 의미하는 수치
- 특정 단어가 등장한 횟수 / 문서의 전체 단어 수

## DTM ~~(와 씨발 DeskTop Music 아시는구나)~~
- 단어의 등장 빈도를 문서별로 표현하는 형식

## IDF
- 단어가 다른 문서들에서 얼마나 드물게나타나고 있는지를 측정한다.
- 어떤 특정 단어가 전체 문서에서 흔하게 등장하면, 해당 단어는 특이성이 낮아 중요도가 낮다고 볼 수 있다.
- 반대로 특정 주제에만 나타나는 단어일수록 그 문서에서의 중요성이 높아지게 된다.

# BM25

- TF-IDF(총 문서의 개소와 단어의 등장 빈도수를 고려하는 알고리즘) 를 기반으로 문서의 길이까지 문서와 쿼리 간 관련성까지 평가한다.
- 일정 빈도 이상에서는 관련성을 천천히 증가시키는 포화 효과를 통해 보정 작업을 거친다.
- 길이가 긴 문서인 경우 짧은 문서에 비해 단어가 많기 때문에 상대적으로 TF가 높을 수 있으므로, 문서의 길이에 대한 보정식도 적용한다.
- 주로 검색 엔진이나 정보 검색 시스템에서 정밀한 결과를 제공하여 사용자에게 더욱 적절한 문서를 추천할 수 있다.

In [18]:
# BM25 Score 구하기 실습

# 1. 관련 라이브러리 불러오기

from rank_bm25 import BM25Okapi
import pandas as pd
from kiwipiepy import Kiwi

In [19]:
# 2. 문서 준비

documents = [
    '고양이와 강아지', # 문서 1.
    '우리 강아지가 다른 강아지를 쫓아갔어', # 문서 2.
    '강아지는 창 밖을 봐' # 문서 3.
]

In [20]:
# 3. 한국어 토크ㄴ화를 위한 함수 작성

kiwi = Kiwi() # 한국어 형태소 분석기

def tokenize_korean_text(text): # 문서 내용 토큰화하는 함수
    # 리스트 컴프리헨션(내포)
    # [최종 결과 for 변수명 in 반복가능한 객체]
    return [token.form for token in kiwi.tokenize(text)]

In [21]:
tokenize_korean_text('우리는 RAG 시스템을 공부합니다!') # 함수 호출

['우리', '는', 'RAG', '시스템', '을', '공부', '하', 'ᆸ니다', '!']

In [22]:
# 4. 각 문서를 토큰화 (단어 단위로 분할)
tokenize_documents = []
for doc in documents:
    tokenize_documents.append(tokenize_korean_text(doc))

In [23]:
print(tokenize_documents)

[['고양이', '와', '강아지'], ['우리', '강아지', '가', '다른', '강아지', '를', '쫓아가', '었', '어'], ['강아지', '는', '창', '밖', '을', '보', '어']]


In [24]:
# 5. 불필요한 불용어 제거

def delete_particles(tokenized_docs: list,
                      particles: list=['와', '가', '를', '는', '을']):
    for sentence_list in tokenized_docs:
        for token in sentence_list:
            if token in particles: # 토큰에 담긴 값이 불용어리스트에 포함되었다면
                sentence_list.remove(token)
    return tokenized_docs

In [25]:
print(delete_particles(tokenize_documents)) # 함수 호출 후 프린트

[['고양이', '강아지'], ['우리', '강아지', '다른', '강아지', '쫓아가', '었', '어'], ['강아지', '창', '밖', '보', '어']]


In [26]:
# 6. BM25 알고리즘을사용한 객체 생성

bm25 = BM25Okapi(tokenize_documents, k1=1.5, b=0.75)

In [27]:
# 7. 문서의 모든 단어를 추출하여 고유 단어 목록 생성

terms = [word for doc in tokenize_documents for word in doc] # 모든 단어를 추출
terms

['고양이',
 '강아지',
 '우리',
 '강아지',
 '다른',
 '강아지',
 '쫓아가',
 '었',
 '어',
 '강아지',
 '창',
 '밖',
 '보',
 '어']

In [29]:
terms = set(terms) # 세트 형태로 변환 후 다시 저장 -> 중복 단어가 제거
terms

{'강아지', '고양이', '다른', '밖', '보', '어', '었', '우리', '쫓아가', '창'}

### 파이썬의 컬렉션 자료형(복수)
- 리스트, 튜플, 딕셔너리, 세트(중복 제거, 순서가 없다)

In [30]:
terms = list(terms)
terms

['우리', '쫓아가', '었', '다른', '어', '고양이', '보', '강아지', '밖', '창']

In [31]:
# 8. 각 문서에 대해 BM25 점수를 계산하여 저장

bm25_scores = []
for term in terms:
    term_scores = bm25.get_scores([term]) # BM25 점수구하기
    bm25_scores.append(term_scores) # 결과를 리스트에

In [32]:
bm25_scores

[array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.03326264, 0.0394778 ]),
 array([0.68764988, 0.        , 0.        ]),
 array([0.        , 0.        , 0.49491756]),
 array([0.05485137, 0.05014982, 0.0394778 ]),
 array([0.        , 0.        , 0.49491756]),
 array([0.        , 0.        , 0.49491756])]

In [33]:
# 9. 결과를 데이터프레임으로 변환

bm25_df = pd.DataFrame(bm25_scores, index=terms, columns=['문서1', '문서2', '문서3'])

In [34]:
bm25_df

,문서1,문서2,문서3
우리,0.000000,0.417001,0.000000
쫓아가,0.000000,0.417001,0.000000
었,0.000000,0.417001,0.000000
다른,0.000000,0.417001,0.000000
어,0.000000,0.033263,0.039478
고양이,0.687650,0.000000,0.000000
보,0.000000,0.000000,0.494918
강아지,0.054851,0.050150,0.039478
밖,0.000000,0.000000,0.494918
창,0.000000,0.000000,0.494918


# BM25를 활용한 RAG system - 라마인덱스

In [38]:
# 1. 사용할 문서 불러오기

from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('./data').load_data()
print(f'문서의 총 개수 : {len(documents)}')

문서의 총 개수 : 10


## KeywordTableIndex

- 문서를 기반으로 키워드와 노드의 매핑을 생성하는 인덱스 클래스

In [39]:
# 2. 인텍스 생성 및 DB 구축(벡터 DB)

from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.indices.keyword_table import KeywordTableIndex

text_splitter = TokenTextSplitter(chunk_size=1024, chunk_overlap=20)

index = KeywordTableIndex.from_documents(
    documents=documents,
    text_splitter=text_splitter,
    extract_keyword=tokenize_korean_text, # 불용어 함스
    max_keywords_per_node=10, # 각 청크에서 추출된 상위 n개 키워드 선택
    max_nodes_per_query=10, # 쿼리에 대해 반환할노드 수를 제한
    show_progress=True # 진행 상황을 확인
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting keywords from nodes:   0%|          | 0/220 [00:00<?, ?it/s]

2026-06-15 20:41:47,480 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:48,810 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:50,456 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:51,618 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:52,753 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:54,886 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:56,123 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:57,343 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 20:41:58,353 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "

In [40]:
# 인덱스 로컬 경로 저장
index.storage_context.persist(persist_dir='./index/bm25_index')

In [41]:
# 4. 저장된 인덱스 불러오기

from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir='./index/bm25_index')
index = load_index_from_storage(storage_context)

2026-06-15 21:06:14,993 - INFO - Loading all indices.


In [42]:
index

In [43]:
# 5. BM25 검색기 객체 생성 및 쿼리 검색

from llama_index.retrievers.bm25 import BM25Retriever

bm25_retriever = BM25Retriever.from_defaults(
    index=index, # bm25를 적용할 인덱스
    similarity_top_k=2, # 유사도 기반 상위 k개 결과 반환
    tokenizer=tokenize_korean_text, #kiwi 토크나이저사용
)

2026-06-15 21:11:36,997 - WARNING - The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
2026-06-15 21:11:37,039 - DEBUG - Building index from IDs objects


In [45]:
# 6. 쿼리 검색

from llama_index.core.response.notebook_utils import display_source_node

query = '변동금리'
retrieverd_nodes = bm25_retriever.retrieve(query)
for node in retrieverd_nodes:
    display_source_node(node, source_length=10000)

**Node ID:** 31e9642c-6b52-4ff3-978b-f69e5dc96faa<br>**Similarity:** 0.0<br>**Text:** <?xml version="1.0" encoding="UTF-8"?>
<bookstore>
  <book category="소설">
    <title lang="ko">태백산맥</title>
    <author>조정래</author>
    <year>1986</year>
    <price>35000</price>
  </book>
  <book category="과학">
    <title lang="ko">코스모스</title>
    <author>칼 세이건</author>
    <year>1980</year>
    <price>25000</price>
  </book>
  <book category="자기계발">
    <title lang="ko">7년의 밤</title>
    <author>정유정</author>
    <year>2011</year>
    <price>18000</price>
  </book>
  <book category="역사">
    <title lang="ko">총, 균, 쇠</title>
    <author>재레드 다이아몬드</author>
    <year>1997</year>
    <price>28000</price>
  </book>
</bookstore><br>

**Node ID:** e7e24bfb-1deb-4de6-bf4c-930138acf3d9<br>**Similarity:** 0.0<br>**Text:** 제목: "Node의 이해: AI의 정보 조각 만들기"



1. Node란 무엇일까요?

Node는 큰 문서를 작은 조각으로 나눈 것입니다. 마치 긴 책을 여러 장의 카드로 나누어 정리하는 것과 비슷합니다. 예를 들어, 역사책 한 권을 시대별로 나누어 카드를 만드는 것처럼, 긴 문서를 여러 개의 Node로 나눕니다.



2. Node는 왜 필요할까요?

큰 문서를 그대로 사용하면 AI가 정보를 찾고 이해하는 데 어려움이 있습니다. 마치 학생이 시험 공부를 할 때 두꺼운 교과서를 한 번에 읽는 것보다, 단원별로 나누어 공부하는 것이 더 효과적인 것처럼, AI도 작은 조각으로 나눈 정보를 더 잘 활용할 수 있습니다.



3. Node의 특징은 무엇일까요?

- 적절한 크기: 한 Node는 AI가 한 번에 처리하기 좋은 크기로 만듭니다.

- 문맥 유지: 내용이 잘리더라도 의미가 통하도록 적절히 나눕니다.

- 관계 유지: 원본 문서와의 연결 정보를 보관합니다.

- 메타데이터: 각 Node가 어디서 왔는지, 어떤 내용인지 알 수 있는 정보를 담습니다.



4. Node는 어떻게 만들어질까요?

Node를 만드는 방법은 여러 가지가 있습니다:

- 문장 단위로 나누기: 문장을 기준으로 나눕니다.

- 단락 단위로 나누기: 내용이 바뀌는 단락을 기준으로 나눕니다.

- 길이 기준으로 나누기: 일정한 길이를 기준으로 나눕니다.



5. Node의 활용

Node는 다음과 같은 상황에서 유용하게 사용됩니다:

- 정보 검색: 필요한 정보를 빠르게 찾을 수 있습니다.

- 질문 답변: 관련된 Node만 활용하여 정확한 답변을 만듭니다.

- 정보 요약: 여러 Node의 내용을 모아 요약할 수 있습니다.



6. Node 사용의 장점

- 효율적인 검색: 작은 단위로 나누어져 있어 검색이 빠릅니다.

- 정확한 답변: 필요한 부분만 정확하게 활용할 수 있습니다.

- 메모리 효율: 필요한 Node만 메모리에 불러올 수 있습니다.<br>

In [46]:
# 7. 
from dotenv import load_dotenv

load_dotenv()

True

In [56]:
# 8. LLM 답변 생성
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model='gpt-4o', temperature=0.5)
query_engine = index.as_query_engine()
response = query_engine.query('메타데이터')
print(response)

2026-06-15 21:32:42,716 - INFO - > Starting query: 메타데이터
2026-06-15 21:32:43,502 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-15 21:32:43,519 - INFO - query keywords: ['used', 'keywords', 'question: what is metadata and how is it used in data management?\n\nkeywords: metadata', 'question', 'metadata', 'data management', 'data', 'management']
2026-06-15 21:32:43,520 - INFO - > Extracted keywords: ['data']
2026-06-15 21:32:45,656 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


메타데이터는 데이터를 설명하는 정보를 의미하며, 파일의 속성이나 구조, 생성 날짜, 수정 날짜, 작성자 정보 등을 포함할 수 있습니다. 데이터 파일의 경우, 파일 경로, 파일 포맷, 데이터의 구조 및 내용에 대한 설명 등이 메타데이터의 예가 될 수 있습니다. 예를 들어, 이미지 파일의 메타데이터에는 파일 경로, 해상도, 파일 크기 등이 포함될 수 있으며, CSV 파일의 메타데이터에는 열 이름, 데이터 형식, 생성 날짜 등이 포함될 수 있습니다.
